# House price prediction model search - linear model

Loop and grid search for the optimal house price model using linear model.

In [ ]:
import geopandas as gpd
import libpysal
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, cross_val_predict

In [ ]:
data_folder = "/Users/martinfleis/Library/CloudStorage/OneDrive-UniverzitaKarlova,Přírodovědeckáfakulta/demoland_data"

Load the data

In [3]:
data = gpd.read_parquet(f"{data_folder}/processed/interpolated/all_oa.parquet")

Filter only explanatory variables.

In [4]:
exvars = data.drop(
    columns=[
        "geo_code",
        "geometry",
        "air_quality_index",
        "house_price_index",
        "jobs_accessibility_index",
        "greenspace_accessibility_index",
    ]
)

Specify grid search parameters. We can limit the options based on previous exploration.

In [10]:
parameters = {"learning_rate": (0.05, 0.1), "max_iter": [500], "max_bins": (64, 128)}
parameters = {}

Define the simple weights matrices.

In [6]:
queen = libpysal.weights.Queen.from_dataframe(data)
weights = {
    "queen": queen,
    "queen2": libpysal.weights.higher_order(queen, k=2, lower_order=True),
    "queen3": libpysal.weights.higher_order(queen, k=3, lower_order=True),
    "queen4": libpysal.weights.higher_order(queen, k=4, lower_order=True),
    "queen5": libpysal.weights.higher_order(queen, k=5, lower_order=True),
    "500m": libpysal.weights.DistanceBand.from_dataframe(data, 500),
    "1000m": libpysal.weights.DistanceBand.from_dataframe(data, 1000),
    "2000m": libpysal.weights.DistanceBand.from_dataframe(data, 2000),
}

/var/folders/n0/xbnn2f2d0ybbp2c_bf_9ztzw0000gn/T/ipykernel_66777/2015151665.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  queen = libpysal.weights.Queen.from_dataframe(data)
/Users/martinfleis/dev/demoland-project/.pixi/envs/default/lib/python3.15/site-packages/libpysal/weights/contiguity.py:354: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
/Users/martinfleis/dev/demoland-project/.pixi/envs/default/lib/python3.15/site-packages/libpysal/weights/util.py:544: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  return W(neighbors=d, **kwargs)
/Users/martinfleis/dev/demoland-project/.pixi/envs/default/lib/python3.15/site-packages/libpysal/weights/util.py:819: UserWarning: The weights matrix is not fully connected: 
 There are 110 disconne

Add combined weights on top.

In [7]:
weights["queen500m"] = libpysal.weights.w_union(weights["queen"], weights["500m"])
weights["queen1000m"] = libpysal.weights.w_union(weights["queen"], weights["1000m"])
weights["queen2000m"] = libpysal.weights.w_union(weights["queen"], weights["2000m"])

Get a mask to ignore missing values.

In [8]:
mask = data.house_price_index.notna()

Use Grid Search CV to find the best model for each weights option.

In [11]:
meta = {}
for name, W in weights.items():
    W.transform = "r"
    exvars = data.drop(
        columns=[
            "geo_code",
            "geometry",
            "air_quality_index",
            "house_price_index",
            "jobs_accessibility_index",
            "greenspace_accessibility_index",
        ]
    )
    for col in exvars.columns.copy():
        exvars[f"{col}_lag"] = libpysal.weights.spatial_lag.lag_spatial(W, exvars[col])
    regressor_lag = LinearRegression()
    est_lag = GridSearchCV(regressor_lag, parameters, verbose=1)
    est_lag.fit(exvars[mask], np.log(data.house_price_index[mask]))
    meta[name] = {"score": est_lag.best_score_}
    y_pred_lag = cross_val_predict(
        est_lag.best_estimator_, exvars[mask], np.log(data.house_price_index[mask]), cv=5
    )
    pred_lag = pd.Series(y_pred_lag, index=data.index[mask])
    residuals_lag = np.log(data.house_price_index[mask]) - pred_lag
    meta[name]["mse"] = mean_squared_error(np.log(data.house_price_index[mask]), pred_lag)
    meta[name]["me"] = residuals_lag.abs().mean()
    meta[name]["prediction"] = pred_lag
    meta[name]["residuals"] = residuals_lag
    meta[name]["model"] = est_lag.best_estimator_

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
('WARNING: ', 47, ' is an island (no neighbors)')
('WARNING: ', 71, ' is an island (no neighbors)')
('WARNING: ', 72, ' is an island (no neighbors)')
('WARNING: ', 89, ' is an island (no neighbors)')
('WARNING: ', 263, ' is an island (no neighbors)')
('WARNING: ', 361, ' is an island (no neighbors)')
('WARNING: ', 364, ' is an island (no neighbors)')
('WARNING: ', 375, ' is an island (no neighbors)')
('WARNING: ', 376, ' is an island (no neighbors)')
('WARNING: ', 377, ' is an island (no neighbors)')
('WARNING: ', 378, ' is an island (no neighbors)')
('WARNING: ', 541, ' is an island (no neighbors)')
('WARNING: ', 642, ' is an island (no neighbors)')
('WARNING: ', 983, ' is an island (no neighbor

Save evaluation metrics as series.

In [12]:
mse = pd.Series([vals["mse"] for vals in meta.values()], index=meta.keys())
me = pd.Series([vals["me"] for vals in meta.values()], index=meta.keys())
score = pd.Series([vals["score"] for vals in meta.values()], index=meta.keys())

Sort according to MSE. Lower is better.

In [13]:
mse.sort_values()

queen4        0.107271
queen3        0.107693
queen5        0.113684
queen1000m    0.113822
queen2        0.118013
queen2000m    0.119838
2000m         0.121287
1000m         0.122148
500m          0.128871
queen500m     0.129673
queen         0.134428
dtype: float64

Sort according to ME. Lower is better.

In [14]:
me.sort_values()

queen4        0.262748
queen3        0.264138
queen1000m    0.266995
queen5        0.269638
1000m         0.274090
queen2        0.275039
queen2000m    0.276948
2000m         0.277656
queen500m     0.282936
500m          0.284517
queen         0.288670
dtype: float64

Sort according to R2. Higher is better.

In [15]:
score.sort_values()

queen         0.001977
queen500m     0.037170
500m          0.047914
2000m         0.073442
1000m         0.079321
queen2000m    0.088549
queen2        0.111699
queen5        0.142517
queen1000m    0.146984
queen3        0.188141
queen4        0.195964
dtype: float64